In [1]:
%pip install chromadb sentence-transformers pandas

     |████████████████████████████████| 21.7 MB 7.8 MB/s eta 0:00:01
     |████████████████████████████████| 488 kB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 12.0 MB 7.9 MB/s eta 0:00:01
     |████████████████████████████████| 310 kB 10.0 MB/s eta 0:00:01
     |████████████████████████████████| 40 kB 8.7 MB/s  eta 0:00:01
     |████████████████████████████████| 180 kB 8.4 MB/s eta 0:00:01
     |████████████████████████████████| 174 kB 10.1 MB/s eta 0:00:01
     |████████████████████████████████| 48 kB 8.9 MB/s eta 0:00:011
     |████████████████████████████████| 245 kB 8.3 MB/s eta 0:00:01
     |████████████████████████████████| 495 kB 9.9 MB/s eta 0:00:01
     |████████████████████████████████| 472 kB 5.9 MB/s eta 0:00:01
     |████████████████████████████████| 90 kB 7.8 MB/s eta 0:00:01
     |████████████████████████████████| 16.8 MB 9.4 MB/s eta 0:00:01
     |████████████████████████████████| 60 kB 10.1 MB/s ta 0:00:011
     |████████████████████████████████| 56 

In [22]:
import pandas as pd

# 관세법
law_df = pd.read_csv("../output/customs_law_chunks.csv",dtype=str)
law_df["source_type"] = "customs_law"

# 한중 PSR
cn_psr_df = pd.read_csv("../output/korus_cn_psr_chunks.csv",dtype=str)
cn_psr_df["source_type"] = "psr"
cn_psr_df["agreement"] = "KOREA_CHINA_FTA"

# 한미 PSR
us_psr_df = pd.read_csv("../output/korus_psr_chunks.csv",dtype=str)
us_psr_df["source_type"] = "psr"
us_psr_df["agreement"] = "KORUS_FTA"

# 한중 통관
cn_customs_df = pd.read_csv("../output/kr_cn_customs_facilitation.csv",dtype=str)
cn_customs_df["source_type"] = "fta"

# 한미 통관
us_customs_df = pd.read_csv("../output/kr_us_customs_facilitation.csv",dtype=str)
us_customs_df["source_type"] = "fta"

df = pd.concat([
    law_df,
    cn_psr_df,
    us_psr_df,
    cn_customs_df,
    us_customs_df
], ignore_index=True)

print(df.shape)

(6328, 7)


In [23]:
print(df.columns)

Index(['article', 'content', 'source_type', 'code', 'agreement', 'chapter',
       'title'],
      dtype='object')


In [24]:
for col in [
    "article",
    "content",
    "code",
    "agreement",
    "chapter",
    "title",
    "source_type"
]:
    if col not in df.columns:
        df[col] = ""

df = df.fillna("")

In [25]:
df["embedding_text"] = (
    df["article"].astype(str) + "\n" +
    df["title"].astype(str) + "\n" +
    df["code"].astype(str) + "\n" +
    df["content"].astype(str)
)

In [26]:
collection = client.get_or_create_collection(
    "customs_knowledge_v2"
)

In [27]:
for idx, row in df.iterrows():

    collection.add(
        ids=[str(idx)],
        documents=[row["embedding_text"]],
        metadatas=[{
            "source_type": row["source_type"],
            "agreement": row["agreement"],
            "article": row["article"],
            "title": row["title"],
            "code": row["code"]
        }]
    )

In [21]:
result = collection.query(
    query_texts=["사전심사"],
    n_results=10
)

for doc, meta in zip(
    result["documents"][0],
    result["metadatas"][0]
):
    print(meta)
    print(doc[:300])
    print("="*80)

{'article': '', 'code': 506.1, 'title': '', 'source_type': 'psr', 'agreement': 'KOREA_CHINA_FTA'}


506.1
0506.10 
골소(骨素)와 뼈[산(酸)처리한 것으로 
한정한다] 
완전생산기준 
 
 
{'title': '', 'agreement': 'KOREA_CHINA_FTA', 'article': '', 'code': 2814.2, 'source_type': 'psr'}


2814.2
2814.20 
암모니아수 
4 단위 세번변경기준 
 
28.15 
 
수산화나트륨[가성(苛性)소다], 
수산화칼륨[가성(苛性)칼륨], 
과산화나트륨이나 과산화칼륨 
 
 
 
 
수산화나트륨[가성(苛性)소다] 
 
 
 
{'source_type': 'psr', 'article': '', 'agreement': 'KOREA_CHINA_FTA', 'title': '', 'code': 2835.39}


2835.39
2835.39 
기타 
4 단위 세번변경기준 
 
28.36 
 
탄산염, 과산화탄산염(과탄산염), 
상관습(商慣習)상의 탄산암모늄 
(카르밤산암모늄을 함유한 것으로 
한정한다) 
 
 
 
{'code': '', 'article': '제320조(가산세의 세목) 이 법에 따른 가산세는 관세의 세목으로 한다.', 'title': '', 'agreement': '', 'source_type': 'customs_law'}
제320조(가산세의 세목) 이 법에 따른 가산세는 관세의 세목으로 한다.


제320조(가산세의 세목) 이 법에 따른 가산세는 관세의 세목으로 한다.
[전문개정 2010. 12. 30.]
 
{'source_type': 'customs_law', 'agreement': '', 'title': '', 'code': '', 'article': '제177조, 제208조부터 제212조까지 및 제321조를 준용한다.'}
제177조, 제208조부터 제212조까지 및 제321조를 준